# ViT + GPT-2 Encoder-Decoder Training
# Bengali Handwritten Text Recognition

This notebook uses **pretrained models** with an encoder-decoder architecture:
- **ViT Encoder**: Pretrained Vision Transformer (google/vit-base-patch16-224)
- **GPT-2 Decoder**: Pretrained GPT-2 with cross-attention (openai-community/gpt2)

## Key Differences from Original:
- Uses full pretrained models (not just blocks)
- Encoder-decoder architecture with cross-attention
- Much better generalization with less data
- Lower learning rate for fine-tuning

## 1. Install Dependencies

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install transformers opencv-python pillow pandas scikit-learn editdistance tqdm

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Import Libraries

In [ ]:
import os
import sys
import shutil
from pathlib import Path

import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

from transformers import ViTModel, GPT2LMHeadModel, AutoImageProcessor

# Set random seeds
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 3. Configure Paths

**For Kaggle:**
- Upload `GraDeT-HTR` folder as dataset
- Upload `Sample_Small` as dataset
- Update paths below

In [ ]:
IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    BASE_PATH = '/kaggle/input/gradet-htr/GraDeT-HTR'
    DATASET_PATH = '/kaggle/input/sample-small/Sample_Small'
    OUTPUT_DIR = '/kaggle/working'
else:
    BASE_PATH = '/home/mahmud1628/Documents/3-2/CSE 330/Project/GraDeT-HTR'
    DATASET_PATH = '/home/mahmud1628/Documents/3-2/CSE 330/Project/Sample_Small'
    OUTPUT_DIR = '/home/mahmud1628/Documents/3-2/CSE 330/Project/output_vit_gpt2'

sys.path.insert(0, BASE_PATH)
sys.path.insert(0, os.path.join(BASE_PATH, 'BnGraphemizer'))

CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, 'checkpoints')
DATASET_PREPARED_DIR = os.path.join(OUTPUT_DIR, 'dataset_prepared')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(DATASET_PREPARED_DIR, exist_ok=True)
os.makedirs(os.path.join(DATASET_PREPARED_DIR, 'images'), exist_ok=True)
os.makedirs(os.path.join(DATASET_PREPARED_DIR, 'labels'), exist_ok=True)

print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"Output Directory: {OUTPUT_DIR}")

## 4. Dataset Preparation

Collect word images from Sample_Small dataset.

In [ ]:
def collect_word_images_from_txt(dataset_path, verbose=False):
    import pandas as pd
    
    words_path = os.path.join(dataset_path, 'Segmentation_Images', 'Words')
    gt_path = os.path.join(dataset_path, 'Recognition_Ground_Truth_Texts')
    data = []
    
    for page_id in sorted(os.listdir(words_path)):
        page_dir = os.path.join(words_path, page_id)
        if not os.path.isdir(page_dir):
            continue
        
        excel_file = os.path.join(gt_path, page_id, f"{page_id}.xlsx")
        if not os.path.exists(excel_file):
            continue
        
        try:
            # ✅ FIX: Read only Sheet 1 (index 0)
            df = pd.read_excel(excel_file, sheet_name=0)  # or sheet_name='Sheet1'
            
            if len(df.columns) >= 2:
                filename_col = df.columns[0]
                text_col = df.columns[1]
                gt_mapping = {f"{str(row[filename_col])}.jpg": str(row[text_col]) 
                             for idx, row in df.iterrows()}
            else:
                continue
        except Exception as e:
            if verbose:
                print(f"Error reading {excel_file}: {e}")
            continue
        
        for line_id in sorted(os.listdir(page_dir)):
            line_dir = os.path.join(page_dir, line_id)
            if not os.path.isdir(line_dir):
                continue
            
            for word_id in sorted(os.listdir(line_dir)):
                word_dir = os.path.join(line_dir, word_id)
                if not os.path.isdir(word_dir):
                    continue
                
                for img_file in sorted(os.listdir(word_dir)):
                    if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                        img_name_without_ext = os.path.splitext(img_file)[0]
                        parts = img_name_without_ext.split('_')
                        
                        if len(parts) < 4:
                            continue
                        
                        img_path = os.path.join(word_dir, img_file)
                        transcription = gt_mapping.get(img_file, "")
                        
                        if not transcription or transcription == 'nan':
                            continue
                        
                        data.append((img_path, transcription, img_file))
    
    return data

print("Collecting word images...")
word_data = collect_word_images_from_txt(DATASET_PATH, verbose=True)  # ✅ Enable verbose
print(f"Total word images: {len(word_data)}")

if len(word_data) > 0:
    print("\nFirst 5 samples:")
    for img_path, text, img_name in word_data[:5]:
        print(f"  {img_name}: '{text}'")
else:
    print("⚠️ WARNING: No data collected! Check dataset structure.")

In [ ]:
# Copy and prepare dataset
print("Preparing dataset...")
csv_data = []

for idx, (src_img_path, text, img_name) in enumerate(tqdm(word_data)):
    new_img_name = f"word_{idx:05d}.jpg"
    dst_img_path = os.path.join(DATASET_PREPARED_DIR, 'images', new_img_name)
    shutil.copy2(src_img_path, dst_img_path)
    csv_data.append({'image_id': new_img_name, 'text': text})

df = pd.DataFrame(csv_data)
csv_path = os.path.join(DATASET_PREPARED_DIR, 'labels', 'label.csv')
df.to_csv(csv_path, index=False, encoding='utf-8')

print(f"Dataset prepared: {len(csv_data)} images")
print(f"\nFirst 5 rows:")
print(df.head())

## 5. Tokenizer Classes

In [ ]:
class TrieTokenizer:
    def __init__(self, vocab, separator=""):
        self.vocab = vocab
        self.separator = separator
        self.trie = self._make_trie()
        
    def _make_trie(self):
        trie = {}
        for token in self.vocab:
            self._add_token(trie, token)
        return trie
    
    def _add_token(self, trie, token):
        node = trie
        for char in token:
            if char not in node:
                node[char] = {}
            node = node[char]
        node[''] = token
    
    def tokenize(self, text):
        tokens = []
        i = 0
        while i < len(text):
            token, length = self._get_next_token(text, i)
            if token:
                tokens.append(token)
                i += length
            else:
                i += 1
        return tokens
    
    def _get_next_token(self, text, start):
        node = self.trie
        last_token = None
        last_length = 0
        
        for i, char in enumerate(text[start:]):
            if char not in node:
                break
            node = node[char]
            if '' in node:
                last_token = node['']
                last_length = i + 1
        
        return last_token, last_length

print("✓ TrieTokenizer defined")

In [ ]:
import unicodedata
from collections import defaultdict
from typing import Union, List

class GraphemeTokenizer:
    def __init__(self, tokenizer_class, max_len=64, separator="", blank_token="_", 
                 oov_token="▁", normalize_unicode=False, normalization_mode="NFKC",
                 normalizer="unicode", printer=print, bos_token="_", eos_token="_",
                 add_bos_token=True, add_eos_token=True):
        self.vocab = list(dict.fromkeys([oov_token, blank_token, bos_token, eos_token]))
        self.max_len = max_len
        self.oov_token = oov_token
        self.blank_token = blank_token
        self.bos_token = bos_token
        self.eos_token = eos_token
        self.add_bos_token = add_bos_token
        self.add_eos_token = add_eos_token
        self.tokenizer_class = tokenizer_class
        self.separator = separator
        self.tokenizer = self.tokenizer_class([(idx) for idx in self.vocab], separator=self.separator)
        self.word2index = {token: idx for idx, token in enumerate(self.vocab)}
        self.normalize_unicode = normalize_unicode
        self.normalization_mode = normalization_mode
        self.print = printer
        self.out_of_vocabulary_info = defaultdict(set)
        self.frequency_counter = defaultdict(int)
        self._set_normalizer(normalizer)
        self.pad_token_id = self.word2index[self.blank_token]
        self.bos_token_id = self.word2index[self.bos_token]
        self.eos_token_id = self.word2index[self.eos_token]

    def tokenize(self, text, padding=False, normalize_unicode=None, normalization_mode=None):
        if isinstance(text, list):
            return [self.tokenize(_text, padding, normalize_unicode, normalization_mode) for _text in text]
        
        normalize_unicode = self.normalize_unicode if normalize_unicode is None else normalize_unicode
        if normalize_unicode:
            text = self._unicode_normalizer(text, normalization_mode)
        
        tokens = self.tokenizer.tokenize(text)
        
        if self.add_bos_token and self.add_eos_token:
            tokens = [self.bos_token] + tokens + [self.eos_token]
        elif self.add_bos_token:
            tokens = [self.bos_token] + tokens
        elif self.add_eos_token:
            tokens = tokens + [self.eos_token]
        
        tokens = tokens[:self.max_len]
        n_tokens = len(tokens)
        
        if padding:
            tokens = tokens + [self.blank_token] * (self.max_len - n_tokens)
        
        tokens_id = [self.word2index.get(token, self.word2index[self.oov_token]) for token in tokens]
        attention_mask = [1] * n_tokens + [0] * (len(tokens) - n_tokens)
        
        return {'tokens': tokens, 'input_ids': tokens_id, 'token_len': n_tokens, 'attention_mask': attention_mask}

    def add_tokens(self, vocab, normalize_unicode=None, reset_oov=False):
        normalize_unicode = self.normalize_unicode if normalize_unicode is None else normalize_unicode
        vocab = self._validate_tokens(vocab, normalize_unicode)
        self.vocab = self.vocab + vocab
        self.tokenizer = self.tokenizer_class([(v) for v in self.vocab], separator=self.separator)
        self.word2index = {token: idx for idx, token in enumerate(self.vocab)}
        self.bos_token_id = self.word2index[self.bos_token]
        self.eos_token_id = self.word2index[self.eos_token]
        if reset_oov:
            self.reset_out_of_vocabulary_info(keys=vocab)

    def _validate_tokens(self, vocab, normalize_unicode=False):
        if normalize_unicode:
            vocab = list(map(self._unicode_normalizer, vocab))
        vocab = sorted(list(set(vocab)))
        vocab = [v for v in vocab if v not in self.vocab]
        return vocab

    def _set_normalizer(self, type="unicode"):
        if type == "unicode":
            self.normalizer = lambda text, mode: unicodedata.normalize(mode, text)
        else:
            self.normalizer = lambda text, mode: text

    def _unicode_normalizer(self, text, mode=None):
        mode = self.normalization_mode if mode is None else mode
        text = self.normalizer(text, mode)
        text = text.replace("\u200c", "").replace("\u200d", "")
        return text

    def ids_to_token(self, ids):
        if not ids:
            raise ValueError("ids must be non-empty")
        if not isinstance(ids[0], list):
            token_list = [self.vocab[idx] for idx in ids 
                         if self.vocab[idx] not in [self.blank_token, self.bos_token, self.eos_token]]
            return token_list
        if isinstance(ids[0], list):
            return list(map(self.ids_to_token, ids))

    def ids_to_text(self, ids):
        if not ids:
            raise ValueError("ids must be non-empty")
        tokens = self.ids_to_token(ids)
        if not isinstance(tokens[0], list):
            return "".join(tokens)
        if isinstance(tokens[0], list):
            return list(map("".join, tokens))

    def _get_index(self, text, token):
        index = self.word2index.get(token)
        self.frequency_counter[token] += 1
        if index is not None:
            return index
        self.out_of_vocabulary_info[token].add(text)
        return self.word2index[self.oov_token]

    def reset_out_of_vocabulary_info(self, keys=None):
        if isinstance(keys, list):
            for k in keys:
                self.out_of_vocabulary_info.pop(k, None)
            return
        if isinstance(keys, str):
            if keys.lower() == "all":
                self.out_of_vocabulary_info = defaultdict(set)
            return

print("✓ GraphemeTokenizer defined")

## 6. Bengali Text Processor

In [ ]:
class BnGraphemizerProcessor:
    def __init__(self, grapheme_file, model_max_length=128, normalize_unicode=True,
                 normalization_mode='NFKC', normalizer="unicode", blank_token="_",
                 bos_token="<s>", eos_token="</s>", add_bos_token=True, add_eos_token=True):
        self.grapheme_file = grapheme_file
        self.model_max_length = model_max_length
        self.blank_token = blank_token
        self.bos_token = bos_token
        self.eos_token = eos_token
        self.list_of_graphemes = self._load_graphemes()
        self.bn_graphmemizer = self._initialize_graphemizer()
        self.pad_token_id = self.bn_graphmemizer.pad_token_id
        self.bos_token_id = self.bn_graphmemizer.bos_token_id
        self.eos_token_id = self.bn_graphmemizer.eos_token_id
        self.vocab = self.bn_graphmemizer.vocab

    def _load_graphemes(self):
        with open(self.grapheme_file, 'r', encoding='utf-8') as f:
            graphemes = sorted(list(set([line.rstrip('\n\r') for line in f.readlines() if line.strip()])))
        return graphemes

    def _initialize_graphemizer(self):
        graphemizer = GraphemeTokenizer(
            tokenizer_class=TrieTokenizer, max_len=self.model_max_length,
            blank_token=self.blank_token, bos_token=self.bos_token, eos_token=self.eos_token,
            add_bos_token=True, add_eos_token=True
        )
        graphemizer.add_tokens(self.list_of_graphemes, reset_oov=True)
        return graphemizer

    def __call__(self, texts, padding=False):
        bng_text_inputs = self.bn_graphmemizer.tokenize(texts, padding=padding)
        bng_inputs = self._get_tokenized_inputs(bng_text_inputs)
        bng_input_ids = torch.Tensor(bng_inputs['input_ids']).long()
        bng_attention_mask = torch.Tensor(bng_inputs['attention_mask']).long()
        if bng_input_ids.ndim == 1:
            bng_input_ids = bng_input_ids.unsqueeze(0)
        if bng_attention_mask.ndim == 1:
            bng_attention_mask = bng_attention_mask.unsqueeze(0)
        return {'input_ids': bng_input_ids, 'attention_mask': bng_attention_mask}

    def _get_tokenized_inputs(self, inputs):
        if not isinstance(inputs, list):
            return {'input_ids': inputs['input_ids'], 'attention_mask': inputs['attention_mask']}
        input_ids, attention_mask = [], []
        for input in inputs:
            if isinstance(input, list):
                input = self._get_tokenized_inputs(input)
            input_ids.append(input['input_ids'])
            attention_mask.append(input['attention_mask'])
        return {'input_ids': input_ids, 'attention_mask': attention_mask}

    def decode(self, input_ids):
        if isinstance(input_ids, torch.Tensor):
            input_ids = input_ids.cpu().numpy() if input_ids.is_cuda else input_ids.numpy()
        if isinstance(input_ids, np.ndarray):
            input_ids = input_ids.tolist()
        if isinstance(input_ids, list):
            if len(input_ids) == 0:
                return ""
            if isinstance(input_ids[0], list):
                return [self.decode(ids) for ids in input_ids]
            else:
                token_list = self.bn_graphmemizer.ids_to_token(input_ids)
                return ''.join(token_list)

print("✓ BnGraphemizerProcessor defined")

## 7. ViT-GPT2 Encoder-Decoder Model

Uses pretrained ViT encoder and GPT-2 decoder with cross-attention.

In [ ]:
class ViTGPT2EncoderDecoder(nn.Module):
    def __init__(self, vocab_size, max_length=128):
        super().__init__()
        
        # Load pretrained ViT encoder
        self.encoder = ViTModel.from_pretrained('google/vit-base-patch16-224')
        
        # Load pretrained GPT-2 decoder and enable cross-attention
        self.decoder = GPT2LMHeadModel.from_pretrained('openai-community/gpt2')
        self.decoder.config.add_cross_attention = True
        self.decoder.config.is_decoder = True
        
        # Resize token embeddings for Bengali vocab
        self.decoder.resize_token_embeddings(vocab_size)
        
        # Project ViT hidden states to GPT-2 hidden size
        vit_hidden_size = self.encoder.config.hidden_size  # 768
        gpt2_hidden_size = self.decoder.config.n_embd      # 768
        
        if vit_hidden_size != gpt2_hidden_size:
            self.encoder_projection = nn.Linear(vit_hidden_size, gpt2_hidden_size)
        else:
            self.encoder_projection = nn.Identity()
        
        self.vocab_size = vocab_size
        self.max_length = max_length
    
    def forward(self, pixel_values, input_ids, attention_mask=None, labels=None):
        # Encode image
        encoder_outputs = self.encoder(pixel_values=pixel_values)
        encoder_hidden_states = encoder_outputs.last_hidden_state  # (batch, 197, 768)
        encoder_hidden_states = self.encoder_projection(encoder_hidden_states)
        
        # Decoder attention mask for cross-attention
        encoder_attention_mask = torch.ones(
            encoder_hidden_states.shape[:2], 
            dtype=torch.long, 
            device=encoder_hidden_states.device
        )
        
        # Decode with cross-attention
        outputs = self.decoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            encoder_hidden_states=encoder_hidden_states,
            encoder_attention_mask=encoder_attention_mask,
            labels=labels,
            return_dict=True
        )
        
        # Calculate accuracy if labels provided
        accuracy = None
        if labels is not None:
            shift_logits = outputs.logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            predictions = torch.argmax(shift_logits, dim=-1)
            
            if attention_mask is not None:
                mask = attention_mask[..., 1:].reshape(-1)
                matches = (predictions.view(-1) == shift_labels.view(-1)).float()
                accuracy = (mask * matches).sum() / mask.sum()
            else:
                accuracy = (predictions == shift_labels).float().mean()
        
        return {
            'loss': outputs.loss,
            'logits': outputs.logits,
            'accuracy': accuracy
        }
    
    @torch.no_grad()
    def generate(self, pixel_values, max_length=128, num_beams=1, bos_token_id=1, eos_token_id=2, pad_token_id=0):
        # Encode image
        encoder_outputs = self.encoder(pixel_values=pixel_values)
        encoder_hidden_states = encoder_outputs.last_hidden_state
        encoder_hidden_states = self.encoder_projection(encoder_hidden_states)
        
        encoder_attention_mask = torch.ones(
            encoder_hidden_states.shape[:2],
            dtype=torch.long,
            device=encoder_hidden_states.device
        )
        
        # Generate with cross-attention
        batch_size = pixel_values.shape[0]
        input_ids = torch.full(
            (batch_size, 1), 
            bos_token_id, 
            dtype=torch.long, 
            device=pixel_values.device
        )
        
        generated = self.decoder.generate(
            input_ids=input_ids,
            encoder_hidden_states=encoder_hidden_states,
            encoder_attention_mask=encoder_attention_mask,
            max_length=max_length,
            num_beams=num_beams,
            eos_token_id=eos_token_id,
            pad_token_id=pad_token_id,
            early_stopping=True
        )
        
        return generated

print("✓ ViTGPT2EncoderDecoder model defined")

## 8. Dataset and DataLoader

In [ ]:
# Initialize text processor
VOCAB_FILE = os.path.join(BASE_PATH, 'tokenization', 'bn_grapheme_1296_from_bengali.ai.buet.txt')
text_processor = BnGraphemizerProcessor(VOCAB_FILE, model_max_length=128)
image_processor = AutoImageProcessor.from_pretrained(
    'google/vit-base-patch16-224',
    size={'height': 224, 'width': 224}
)

print(f"Vocabulary size: {len(text_processor.vocab)}")
print(f"PAD token ID: {text_processor.pad_token_id}")
print(f"BOS token ID: {text_processor.bos_token_id}")
print(f"EOS token ID: {text_processor.eos_token_id}")

In [ ]:
class HandwrittenDataset(Dataset):
    def __init__(self, images_dir, data_frame, image_processor, text_processor):
        super().__init__()
        self.images_dir = images_dir
        self.df = data_frame
        self.image_processor = image_processor
        self.text_processor = text_processor
        self.image_ids = self.df["image_id"].values
        self.texts = self.df["text"].values
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, index):
        image = Image.open(os.path.join(self.images_dir, str(self.image_ids[index]))).convert('RGB')
        text = str(self.texts[index])
        
        # Process image
        image_inputs = self.image_processor(image, return_tensors="pt")
        pixel_values = image_inputs['pixel_values'][0]
        
        # Process text
        text_inputs = self.text_processor(text, padding=True)
        input_ids = text_inputs['input_ids'][0]
        attention_mask = text_inputs['attention_mask'][0]
        
        return {
            'pixel_values': pixel_values,
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': input_ids.clone()
        }

# Split dataset
images_dir = os.path.join(DATASET_PREPARED_DIR, 'images')
labels_file = os.path.join(DATASET_PREPARED_DIR, 'labels', 'label.csv')
df = pd.read_csv(labels_file)

train_df, val_df = train_test_split(df, test_size=0.15, random_state=SEED)
train_dataset = HandwrittenDataset(images_dir, train_df, image_processor, text_processor)
val_dataset = HandwrittenDataset(images_dir, val_df, image_processor, text_processor)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

# DataLoaders
BATCH_SIZE = 8
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Batch size: {BATCH_SIZE}")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

## 9. Initialize Model

In [ ]:
# Clear GPU memory
import gc
if 'model' in globals():
    del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ViTGPT2EncoderDecoder(vocab_size=len(text_processor.vocab), max_length=128)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✓ Model initialized on {device}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size: {total_params * 4 / 1e6:.2f} MB")

## 10. Training Setup

**Lower learning rate for fine-tuning pretrained models!**

In [ ]:
EPOCHS = 15
LEARNING_RATE = 5e-5  # Lower LR for fine-tuning
USE_AMP = True

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
scaler = torch.cuda.amp.GradScaler() if USE_AMP and torch.cuda.is_available() else None

print(f"Epochs: {EPOCHS}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Optimizer: AdamW with weight decay")
print(f"Mixed precision: {USE_AMP and torch.cuda.is_available()}")

## 11. Training Loop

In [ ]:
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print("Starting training...\n")

for epoch in range(EPOCHS):
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print("-" * 50)
    
    # Training
    model.train()
    train_loss = 0
    train_acc = 0
    
    pbar = tqdm(train_loader, desc="Training")
    for batch in pbar:
        pixel_values = batch['pixel_values'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        if scaler:
            with torch.cuda.amp.autocast():
                outputs = model(pixel_values, input_ids, attention_mask, labels)
                loss = outputs['loss']
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(pixel_values, input_ids, attention_mask, labels)
            loss = outputs['loss']
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
        train_loss += loss.item()
        train_acc += outputs['accuracy'].item()
        pbar.set_postfix({'loss': loss.item(), 'acc': outputs['accuracy'].item()})
    
    avg_train_loss = train_loss / len(train_loader)
    avg_train_acc = train_acc / len(train_loader)
    
    # Validation
    model.eval()
    val_loss = 0
    val_acc = 0
    
    with torch.no_grad():
        for batch in val_loader:
            pixel_values = batch['pixel_values'].to(device)
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(pixel_values, input_ids, attention_mask, labels)
            val_loss += outputs['loss'].item()
            val_acc += outputs['accuracy'].item()
    
    avg_val_loss = val_loss / len(val_loader)
    avg_val_acc = val_acc / len(val_loader)
    
    scheduler.step(avg_val_loss)
    
    history['train_loss'].append(avg_train_loss)
    history['train_acc'].append(avg_train_acc)
    history['val_loss'].append(avg_val_loss)
    history['val_acc'].append(avg_val_acc)
    
    print(f"  Train Loss: {avg_train_loss:.4f}, Train Acc: {avg_train_acc:.4f}")
    print(f"  Val Loss: {avg_val_loss:.4f}, Val Acc: {avg_val_acc:.4f}\n")
    
    if (epoch + 1) % 5 == 0:
        checkpoint_path = os.path.join(CHECKPOINT_DIR, f'checkpoint_epoch_{epoch+1}.pt')
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_val_loss,
        }, checkpoint_path)
        print(f"Checkpoint saved: {checkpoint_path}")

print("Training completed!")

## 12. Plot Training History

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(history['train_loss'], label='Train Loss', marker='o')
ax1.plot(history['val_loss'], label='Val Loss', marker='s')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True)

ax2.plot(history['train_acc'], label='Train Accuracy', marker='o')
ax2.plot(history['val_acc'], label='Val Accuracy', marker='s')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training and Validation Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_history.png'), dpi=300)
plt.show()

print(f"Final Train Acc: {history['train_acc'][-1]:.4f}")
print(f"Final Val Acc: {history['val_acc'][-1]:.4f}")

## 13. Test Inference

In [ ]:
model.eval()
num_samples = 5

print("Sample Predictions:\n")
print("=" * 80)

for i in range(min(num_samples, len(val_dataset))):
    sample = val_dataset[i]
    pixel_values = sample['pixel_values'].unsqueeze(0).to(device)
    
    with torch.no_grad():
        generated_ids = model.generate(
            pixel_values=pixel_values,
            max_length=128,
            num_beams=3,
            bos_token_id=text_processor.bos_token_id,
            eos_token_id=text_processor.eos_token_id,
            pad_token_id=text_processor.pad_token_id
        )
    
    predicted_text = text_processor.decode(generated_ids[0].cpu().numpy())
    ground_truth = text_processor.decode(sample['labels'].cpu().numpy())
    
    print(f"Sample {i+1}:")
    print(f"  Ground Truth: {ground_truth}")
    print(f"  Predicted:    {predicted_text}")
    print(f"  Match: {'✓' if predicted_text.strip() == ground_truth.strip() else '✗'}")
    print("-" * 80)

print("\n✓ Inference completed!")